In [ ]:
import os 
os.getcwd()
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os

In [ ]:

# Load the filez
data = loadmat('data.mat')
illumination = loadmat("illumination.mat")
pose=loadmat("pose.mat")
# See all the keys (variables stored in the .mat file)
print(f"data keys are {data.keys()}")
print(f"illumination keys are {illumination.keys()}")
print(f"pose keys are {pose.keys()}")

In [ ]:
n_2=np.arange(0,68,1)
n_2p=np.arange(0,13,1)
pose_classified={}
for i in n_2:
    imgs=[]
    for j in n_2p:
        imgs.append(pose["pose"][:,:,j,i])
    pose_classified[i]=imgs

In [ ]:
n_3=np.arange(0,68,1)
n_3i=np.arange(0,21,1)
illum_classified={}
for i in n_3:
    imgs=[]
    for j in n_3i:
        imgs.append(illumination["illum"][:,j,i])
    illum_classified[i]=imgs

### data visualization 

In [ ]:
n=np.arange(1,201,1)
data_classified = {}
for i in n:
    imgs=[]
    for j in range(3):
        imgs.append(data["face"][:,:,3*i-3+j])
    data_classified[i]=imgs

In [ ]:
len(data_classified[1])

In [ ]:
fig, ax= plt.subplots(1,3)
ax[0].imshow(data_classified[90][0], cmap="gray") 
ax[0].set_title("neutral")
ax[0].axis("off")

ax[1].imshow(data_classified[90][1], cmap="gray")
ax[1].set_title("expression")
ax[1].axis("off")

ax[2].imshow(data_classified[90][2], cmap="gray")
ax[2].set_title("illumination")
ax[2].axis("off")

# data + labeling

In [ ]:
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])
person_labels = np.repeat(np.arange(200), 3) #for task1

# for task2:
neutral_indices = list(range(0, 600, 3))
expression_indices = list(range(1, 600, 3))
binary_labels = np.zeros(600, dtype=int)
binary_labels[expression_indices] = 1


### PCA

In [ ]:
# Step 1: Flatten the face data
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])

# Step 2: Calculate the mean and center the data
mean = np.mean(flattened_data, axis=0)
centered_data = flattened_data - mean

original_shape = data["face"][:, :, 0].shape
print(f"Original image shape: {original_shape}")

# Now we know the correct dimensions for reshape
face_height, face_width = original_shape

# Step 3: Calculate the covariance matrix
cov_matrix = np.cov(centered_data, rowvar=False)

# Step 4: Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Step 5: Sort eigenvalues and corresponding eigenvectors in descending order
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Step 6: Calculate explained variance and cumulative variance
explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
cumulative_variance = np.cumsum(explained_variance_ratio)

# Step 7: Select top k components (for example, top components that explain 95% of variance)
k = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Number of components needed to explain 95% of variance: {k}")

# Step 8: Get the principal components
principal_components = eigenvectors[:, :k]

# Step 9: Project the original data onto the principal components
pca_result = np.dot(centered_data, principal_components)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'b--o')
plt.axhline(y=0.95, color='r', linestyle='-', label='95% cut-off threshold')
plt.title('The number of components needed to explain variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative variance (%)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Choose a sample face from data_classified
sample_idx = 90  # You can change this to any index within your dataset
expression_idx = 1  # 0 for neutral, 1 for expression, 2 for illumination

# Create a figure with two subplots side by side
plt.figure(figsize=(12, 5))

# Plot the original face image
plt.subplot(1, 2, 1)
original_face = data_classified[sample_idx][expression_idx]
plt.imshow(original_face, cmap='gray')
plt.title(f'Original Image ({"neutral" if expression_idx == 0 else "expression" if expression_idx == 1 else "illumination"})')
plt.axis('off')

# Flatten the original face for PCA projection
flattened_face = original_face.flatten()

# Project the face into the PCA space
n_components = 90  # Adjust based on your preference
reduced_components = eigenvectors[:, :n_components]
projected = np.dot(flattened_face - mean, reduced_components)

# Reconstruct from the reduced space
reconstructed = np.dot(projected, reduced_components.T) + mean
reconstructed_face = reconstructed.reshape(original_face.shape)

# Plot the PCA-reconstructed image
plt.subplot(1, 2, 2)
plt.imshow(reconstructed_face, cmap='gray')
plt.title(f'PCA Reconstruction ({n_components} components)')
plt.axis('off')

plt.suptitle('Original vs PCA-Reconstructed Face Image')
plt.tight_layout()
plt.show()

In [ ]:
# Create a figure with 3 rows and 2 columns
plt.figure(figsize=(12, 15))
expression_names = ["neutral", "expression", "illumination"]

for i in range(3):
    # Plot the original face image
    plt.subplot(3, 2, 2*i+1)
    original_face = data_classified[sample_idx][i]
    plt.imshow(original_face, cmap='gray')
    plt.title(f'Original ({expression_names[i]})')
    plt.axis('off')
    
    # Flatten the original face for PCA projection
    flattened_face = original_face.flatten()
    
    # Project and reconstruct
    projected = np.dot(flattened_face - mean, reduced_components)
    reconstructed = np.dot(projected, reduced_components.T) + mean
    reconstructed_face = reconstructed.reshape(original_face.shape)
    
    # Plot the reconstructed image
    plt.subplot(3, 2, 2*i+2)
    plt.imshow(reconstructed_face, cmap='gray')
    plt.title(f'PCA ({n_components} components)')
    plt.axis('off')

plt.suptitle('Original vs PCA-Reconstructed Face Images')
plt.tight_layout()
plt.show()

# PCA_funciton

In [ ]:
import numpy as np

def compute_pca(data_matrix, num_components=None, variance_threshold=None):
    """
    Perform PCA on a data matrix (samples × features).
    
    Parameters:
        data_matrix: np.ndarray, shape (n_samples, n_features)
        num_components: int or None — number of components to keep
        variance_threshold: float or None — keep components that explain up to this cumulative variance (e.g., 0.95)
    
    Returns:
        pca_result: projected data, shape (n_samples, num_components)
        components: principal components (eigenvectors)
        explained_variance_ratio: array of variance explained by each component
        mean: mean of original data (for inverse transform if needed)
    """
    # Step 1: Center the data
    mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - mean

    # Step 2: Covariance matrix
    cov_matrix = np.cov(centered_data, rowvar=False)

    # Step 3: Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    # Step 4: Sort in descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 5: Compute explained variance
    explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
    cumulative_variance = np.cumsum(explained_variance_ratio)

    # Step 6: Determine number of components
    if variance_threshold is not None:
        num_components = np.argmax(cumulative_variance >= variance_threshold) + 1
        print(f"Using {num_components} components to explain {variance_threshold*100:.1f}% variance.")
    elif num_components is None:
        num_components = data_matrix.shape[1]  # Keep all

    # Step 7: Select components and project
    selected_components = eigenvectors[:, :num_components]
    pca_result = np.dot(centered_data, selected_components)

    return pca_result, selected_components, explained_variance_ratio[:num_components], mean
